# Module 05 — the Δ₂-support predictor: can a Transformer learn *where* a correction matrix has its large elements?

Plan 05 (Δ-probed IR pipeline), Module 05 of the Udacity AI Mastery Capstone. **Task type: sequence modelling with a
Transformer (PyTorch).** A molecule is a sequence of DFT normal-mode tokens; the network predicts, per pair of modes,
whether the correction matrix Δ₂ between two levels of theory has a large element there (the *support* of Δ₂). The
prediction is a learned prior that tells the larger project's recovery where to place its expensive measurements.
**Success criterion (recipe):** the number of probing patterns the prior implies at recall 0.9 against the prior-free
count, and agreement with the prior-free check on a held-out molecule — not classification accuracy.

**Dataset:** Δ₂ = H(ωB97X) − H(B3LYP) per molecule, from (i) the public Hessian QM9 set (Williams et al., 2025; figshare
DOI 10.6084/m9.figshare.26363959, 41,645 ωB97X/6-31G* Hessians, CC0) and (ii) B3LYP/6-31G* Hessians recomputed by this
project's corpus factory (`../corpus/`, four layers, 11,321 candidates; the number actually computed is fixed by a dated
note after the timing test). Computed ab initio data: not synthetic, not AI-generated, not reused from an earlier module.

> **Status of this notebook: skeleton.** Cells marked `FIXTURE` run on the project's benzene dry-run tensor and prove the
> code path only. Cells marked `STUB` raise `NotImplementedError` until the corpus exists. No number below is a result.

Sections: **Load and preprocess · Baseline model · Controlled comparison · Training outputs · Evaluation · Example
behaviour · Summary**.

## 1. Load and preprocess

The corpus builder turns each molecule into (a) a token sequence — one token per DFT normal mode: frequency, family one-hot, symmetry-block id, mode participation by atom type — and (b) a 0/1 label per mode pair: whether |Δ₂,ij| exceeds θ · max_k |Δ₂,kk| (θ = 0.1, recipe). Translation/rotation modes are projected out before Δ₂ is formed (Hessian QM9 stores them).

In [ ]:
# FIXTURE: the benzene dry-run tensor through the same builder the corpus will use
import sys, json
from pathlib import Path
import numpy as np
sys.path.insert(0, str(Path('..').resolve()))
from m05.build_corpus import build_fixture, THETA
build_fixture()                 # writes ../data/corpus_fixture.npz and its manifest (prints the manifest)
z = np.load('../data/corpus_fixture.npz'); names = sorted({k.split('__')[0] for k in z.files})
fx = {'name': names[0], 'tokens': z[f'{names[0]}__tokens'], 'labels': z[f'{names[0]}__labels']}
print(fx['name'], 'tokens', fx['tokens'].shape, 'labels', fx['labels'].shape, 'positive pairs', int(fx['labels'].sum() // 2), 'theta', THETA)

In [ ]:
# STUB: the real corpus (Hessian QM9 vacuum shards + the corpus factory's results) — enabled when the dated subset-size note exists
CORPUS_READY = False
if not CORPUS_READY:
    raise NotImplementedError('corpus not built yet: run ../corpus/run_corpus.py (after the anchor job) and set CORPUS_READY = True')

## 2. Baseline model (PyTorch)

A small encoder-only Transformer over the mode tokens with a pair head: for modes i, j the head reads the two encoded tokens and outputs the logit of 'large element at (i, j)'. Baseline configuration (recipe): d_model 64, 4 heads, 2 layers, dropout 0.1; loss = class-weighted binary cross-entropy over the M(M−1)/2 pairs; optimiser and schedule fixed in the recipe; seeds 0, 1, 2.

In [ ]:
import torch
from m05.model import SupportTransformer, implied_pattern_count
model = SupportTransformer(d_in=fx['tokens'].shape[-1], d_model=64, n_heads=4, n_layers=2, dropout=0.1)
print('parameters:', sum(p.numel() for p in model.parameters()))

## 3. Controlled comparison

One change at a time (recipe; Distilled §5): **baseline (2 layers) versus 4 layers**, everything else fixed, three seeds each. A second comparison is allowed only if the first prints a difference larger than the seed spread.

In [ ]:
# STUB: training loop over the corpus with a leave-molecule-out split by scaffold family; writes ../out/training_log.json
raise NotImplementedError('training waits for the corpus')

## 4. Training outputs

Loss curves (train/validation) per configuration and seed; the implied pattern count at recall 0.9 as a function of epoch.

In [ ]:
# STUB: plots from ../out/training_log.json → figures/loss_curves.png, figures/pattern_count.png
raise NotImplementedError('no training log yet')

## 5. Evaluation

Metrics appropriate to the task: per-pair precision and recall at the operating point recall = 0.9; the **implied pattern count** K_prior against the prior-free count K (the project's own metric); calibration of the pair probabilities (reliability curve). All on held-out molecules; the benzene fixture is never a test molecule.

In [ ]:
# FIXTURE: the metric itself, on the fixture, to show what it computes (not a result)
with torch.no_grad():
    tokens = torch.as_tensor(fx['tokens'], dtype=torch.float32).unsqueeze(0)
    labels = torch.as_tensor(fx['labels'], dtype=torch.float32)
    prob = torch.sigmoid(model(tokens))[0]
print('implied pattern count at recall 0.9 (untrained model, fixture):', implied_pattern_count(prob, labels, recall=0.9))

## 6. Example behaviour

One concrete case each: a molecule where the prior saves patterns; a failure case (a large element the prior misses and what it costs the recovery); overfitting or instability if observed.

In [ ]:
# STUB
raise NotImplementedError('needs trained models')

## 7. Summary

To be written from the outputs above: what the comparison showed, whether the learned prior earns a licence (recipe: pattern saving and agreement with the prior-free check on a held-out molecule), and what the project does with the answer either way.